In [1]:
import os, time, random, gc
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report

from PIL import Image
import torchvision.transforms as T

import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("사용 장치:", device)
if torch.cuda.is_available():
    print("GPU 모델명:", torch.cuda.get_device_name(0))


사용 장치: cuda
GPU 모델명: NVIDIA GeForce RTX 5070


In [2]:
DATA_ROOT = Path(r"C:\\Users\\USER\\Desktop\\대학\\3학년 1학기\\머신러닝 및 실습\\캐글 알츠하이머 - 개인\\alzheimer-prediction")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"
TRAIN_CSV = DATA_ROOT / "train.csv"
SUB_CSV   = DATA_ROOT / "sample_submission.csv"

OUTPUT_DIR = Path("./outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
CKPT_DIR = OUTPUT_DIR / "checkpoints_v2"; CKPT_DIR.mkdir(exist_ok=True)

LABEL2IDX = {'MildDemented': 0, 'ModerateDemented': 1, 'NonDemented': 2, 'VeryMildDemented': 3}
IDX2LABEL = {v: k for k, v in LABEL2IDX.items()}
NUM_CLASSES = 4
SLICES_PER_PATIENT = 61

# 과적합 방지 하이퍼파라미터 셋업
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_EPOCHS = 3          # 암기 방지를 위해 에폭 축소
NUM_EPOCHS_CRT = 1      # cRT 1에폭 제한으로 소수 클래스 과적합 차단
LR_MAX = 3e-4
LR_CRT = 1e-4
WEIGHT_DECAY = 0.01      # 가중치 감쇠(L2 규제) 2배 강화
NUM_FOLDS = 4
NUM_WORKERS = 0
MODEL_NAME = 'convnext_tiny'
DROP_RATE = 0.2         # Head 직전 Dropout 대폭 강화
DROP_PATH_RATE = 0.1    # Stochastic Depth 강화

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


In [3]:
def get_patient_id(fn):
    num = int(fn.replace('train_', '').replace('test_', '').replace('.jpg', ''))
    return (num - 1) // SLICES_PER_PATIENT

def get_slice_idx(fn):
    num = int(fn.replace('train_', '').replace('test_', '').replace('.jpg', ''))
    return (num - 1) % SLICES_PER_PATIENT

train_df_full = pd.read_csv(TRAIN_CSV)
train_df_full['label_idx'] = train_df_full['label'].map(LABEL2IDX)
train_df_full['patient_id'] = train_df_full['filename'].apply(get_patient_id)
train_df_full['slice_idx']  = train_df_full['filename'].apply(get_slice_idx)

# [가설 반영]: 61장 중 뇌실 및 병변 특징이 뚜렷한 정중앙 30장(15~44)만 남기고 양끝 노이즈 컷
train_df = train_df_full[(train_df_full['slice_idx'] >= 15) & (train_df_full['slice_idx'] < 45)].reset_index(drop=True)

sub_df = pd.read_csv(SUB_CSV)
sub_df['patient_id'] = sub_df['filename'].apply(get_patient_id)
sub_df['slice_idx']  = sub_df['filename'].apply(get_slice_idx)

print(f"Train(전체): {len(train_df_full)}장 -> 노이즈 필터링 후: {len(train_df)}장 (속도 2배 향상)")
print(f"Test(제출용): {len(sub_df)}장")
print(train_df['label'].value_counts())

class_counts = train_df['label_idx'].value_counts().sort_index().values


Train(전체): 68808장 -> 노이즈 필터링 후: 33840장 (속도 2배 향상)
Test(제출용): 17629장
label
NonDemented         26400
VeryMildDemented     5340
MildDemented         1980
ModerateDemented      120
Name: count, dtype: int64


In [4]:
class AlzheimerDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / row['filename']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img
        return img, int(row['label_idx'])

def get_train_transform():
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
        T.RandomErasing(p=0.25, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),
    ])

def get_val_transform():
    return T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

def get_tta_transform(flip=False):
    tfms = [T.Resize((IMG_SIZE, IMG_SIZE))]
    if flip:
        tfms.append(T.RandomHorizontalFlip(p=1.0))
    tfms += [T.ToTensor(), T.Normalize(MEAN, STD)]
    return T.Compose(tfms)


In [5]:
class CBFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.9999, gamma=2.0):
        super().__init__()
        cc = np.asarray(class_counts, dtype=np.float64)
        eff = 1.0 - np.power(beta, cc)
        w = (1.0 - beta) / eff
        w = w / w.sum() * len(cc)
        self.register_buffer('weights', torch.tensor(w, dtype=torch.float32))
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        return (self.weights[targets] * focal).mean()

def build_model():
    m = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES,
                          drop_rate=DROP_RATE, drop_path_rate=DROP_PATH_RATE)
    return m.to(device)

def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = ('head' in name)


In [6]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    tl, tn = 0.0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        bs = imgs.size(0)
        tl += loss.item() * bs
        tn += bs
    return tl / tn

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    AL, AY = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast():
            logits = model(imgs)
        AL.append(logits.float().cpu().numpy())
        AY.append(labels.numpy())
    L = np.concatenate(AL, 0); Y = np.concatenate(AY, 0)
    P = L.argmax(1)
    return f1_score(Y, P, average='macro'), L, Y, P

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    A = []
    for imgs in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast():
            logits = model(imgs)
        A.append(logits.float().cpu().numpy())
    return np.concatenate(A, 0)


In [7]:
sgkf = StratifiedGroupKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
folds = []
for tr_idx, va_idx in sgkf.split(train_df, train_df['label_idx'], groups=train_df['patient_id']):
    folds.append((tr_idx, va_idx))

print("=== 환자 단위 교차 분리 검증 장벽 작동 여부 확인 ===")
for fi, (tr, va) in enumerate(folds):
    trp = set(train_df.iloc[tr]['patient_id'])
    vap = set(train_df.iloc[va]['patient_id'])
    print(f"Fold {fi}: Train데이터 수={len(tr)} | Val데이터 수={len(va)} | 중복환자수={len(trp & vap)}명 (무조건 0이어야 데이터누수 차단됨)")


=== 환자 단위 교차 분리 검증 장벽 작동 여부 확인 ===
Fold 0: Train데이터 수=25380 | Val데이터 수=8460 | 중복환자수=0명 (무조건 0이어야 데이터누수 차단됨)
Fold 1: Train데이터 수=25380 | Val데이터 수=8460 | 중복환자수=0명 (무조건 0이어야 데이터누수 차단됨)
Fold 2: Train데이터 수=25380 | Val데이터 수=8460 | 중복환자수=0명 (무조건 0이어야 데이터누수 차단됨)
Fold 3: Train데이터 수=25380 | Val데이터 수=8460 | 중복환자수=0명 (무조건 0이어야 데이터누수 차단됨)


In [8]:
oof_logits = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
oof_labels = np.zeros(len(train_df), dtype=np.int64)
fold_scores = []
t0 = time.time()

for fi, (tr_idx, va_idx) in enumerate(folds):
    print(f"\n{'='*50}\n🚀 FOLD {fi+1}/{NUM_FOLDS} 학습 개시\n{'='*50}")
    tf = time.time()
    tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
    va_df = train_df.iloc[va_idx].reset_index(drop=True)

    tr_ds = AlzheimerDataset(tr_df, TRAIN_DIR, transform=get_train_transform())
    va_ds = AlzheimerDataset(va_df, TRAIN_DIR, transform=get_val_transform())
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
    criterion = CBFocalLoss(class_counts).to(device)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR_MAX, epochs=NUM_EPOCHS, steps_per_epoch=len(tr_loader),
        pct_start=0.1, anneal_strategy='cos')
    scaler = GradScaler()

    print(">> [Phase 1] 전체 가중치 균형 학습 돌입")
    for ep in range(NUM_EPOCHS):
        te = time.time()
        tl = train_one_epoch(model, tr_loader, criterion, optimizer, scheduler, scaler)
        vf, _, _, _ = evaluate(model, va_loader)
        print(f"   Ep {ep+1}/{NUM_EPOCHS} loss={tl:.4f} val_f1={vf:.4f} ({time.time()-te:.1f}s)")

    print(">> [Phase 2] cRT 미세 교정 돌입 (백본 고정, 1:1:1:1 균형 분포 샘플러 가동)")
    freeze_backbone(model)
    tr_labels = tr_df['label_idx'].values
    cls_cnt = np.bincount(tr_labels, minlength=NUM_CLASSES)
    sw = np.array([1.0 / cls_cnt[y] for y in tr_labels], dtype=np.float64)
    sampler = WeightedRandomSampler(sw, num_samples=len(tr_labels), replacement=True)
    tr_loader_bal = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer2 = torch.optim.AdamW(head_params, lr=LR_CRT, weight_decay=WEIGHT_DECAY)
    criterion2 = nn.CrossEntropyLoss()

    best_f1, best_state = -1.0, None
    for ep in range(NUM_EPOCHS_CRT):
        te = time.time()
        tl = train_one_epoch(model, tr_loader_bal, criterion2, optimizer2, None, scaler)
        vf, _, _, _ = evaluate(model, va_loader)
        print(f"   cRT Ep {ep+1}/{NUM_EPOCHS_CRT} loss={tl:.4f} val_f1={vf:.4f} ({time.time()-te:.1f}s)")
        if vf > best_f1:
            best_f1 = vf
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_f1, _, _, _ = evaluate(model, va_loader)

    model.load_state_dict(best_state)
    vf_final, L, Y, P = evaluate(model, va_loader)
    oof_logits[va_idx] = L
    oof_labels[va_idx] = Y
    torch.save(best_state, CKPT_DIR / f"fold{fi}_best.pt")
    fold_scores.append(best_f1)
    print(f"✨ Fold {fi} 최적 가중치 저장 완료 | Best Val Macro F1={best_f1:.4f} ({time.time()-tf:.1f}s)")

    del model, optimizer, optimizer2, scheduler, scaler, tr_loader, va_loader, tr_loader_bal
    torch.cuda.empty_cache(); gc.collect()

print(f"\n🎉 전체 4-Fold 교차 검증 완료 소요 시간: {(time.time()-t0)/60:.1f}분")
print(f"Folds 최종 성적 리스트: {[f'{s:.4f}' for s in fold_scores]}")
print(f"평균치 Macro F1 성적: {np.mean(fold_scores):.4f}")



🚀 FOLD 1/4 학습 개시
>> [Phase 1] 전체 가중치 균형 학습 돌입
   Ep 1/3 loss=0.0487 val_f1=0.4975 (163.2s)
   Ep 2/3 loss=0.0153 val_f1=0.8332 (134.2s)
   Ep 3/3 loss=0.0043 val_f1=0.9287 (136.4s)
>> [Phase 2] cRT 미세 교정 돌입 (백본 고정, 1:1:1:1 균형 분포 샘플러 가동)
   cRT Ep 1/1 loss=0.1295 val_f1=0.9202 (109.5s)
✨ Fold 0 최적 가중치 저장 완료 | Best Val Macro F1=0.9202 (566.4s)

🚀 FOLD 2/4 학습 개시
>> [Phase 1] 전체 가중치 균형 학습 돌입
   Ep 1/3 loss=0.0559 val_f1=0.5705 (142.4s)
   Ep 2/3 loss=0.0157 val_f1=0.8445 (145.8s)
   Ep 3/3 loss=0.0055 val_f1=0.9078 (147.4s)
>> [Phase 2] cRT 미세 교정 돌입 (백본 고정, 1:1:1:1 균형 분포 샘플러 가동)
   cRT Ep 1/1 loss=0.1709 val_f1=0.8830 (105.0s)
✨ Fold 1 최적 가중치 저장 완료 | Best Val Macro F1=0.8830 (562.3s)

🚀 FOLD 3/4 학습 개시
>> [Phase 1] 전체 가중치 균형 학습 돌입
   Ep 1/3 loss=0.0435 val_f1=0.5981 (144.0s)
   Ep 2/3 loss=0.0140 val_f1=0.8332 (142.6s)
   Ep 3/3 loss=0.0052 val_f1=0.8724 (148.2s)
>> [Phase 2] cRT 미세 교정 돌입 (백본 고정, 1:1:1:1 균형 분포 샘플러 가동)
   cRT Ep 1/1 loss=0.1593 val_f1=0.8589 (99.8s)
✨ Fold 2 최적 가중치 저장 완료 | 

In [9]:
oof_preds = oof_logits.argmax(1)
oof_f1 = f1_score(oof_labels, oof_preds, average='macro')
print(f"OOF Macro F1 (단일 슬라이스 단위 기준): {oof_f1:.4f}")
print("\n=== OOF 상세 기하학적 매핑 성적표 ===")
print(classification_report(oof_labels, oof_preds,
      target_names=[IDX2LABEL[i] for i in range(NUM_CLASSES)], digits=4))

oof_df = train_df.copy()
sm = F.softmax(torch.from_numpy(oof_logits), dim=1).numpy()
for c in range(NUM_CLASSES):
    oof_df[f'p{c}'] = sm[:, c]
pat = oof_df.groupby('patient_id')[[f'p{c}' for c in range(NUM_CLASSES)]].mean()
pat_pred = pat.values.argmax(1)
pat_true = oof_df.groupby('patient_id')['label_idx'].first().values
print(f"OOF Macro F1 (환자단위 소프트 보팅 최종 결합 성적): {f1_score(pat_true, pat_pred, average='macro'):.4f}")


OOF Macro F1 (단일 슬라이스 단위 기준): 0.6477

=== OOF 상세 기하학적 매핑 성적표 ===
                  precision    recall  f1-score   support

    MildDemented     0.1903    0.9803    0.3188      1980
ModerateDemented     1.0000    0.7417    0.8517       120
     NonDemented     0.9813    0.6595    0.7889     26400
VeryMildDemented     0.6060    0.6594    0.6316      5340

        accuracy                         0.6786     33840
       macro avg     0.6944    0.7602    0.6477     33840
    weighted avg     0.8759    0.6786    0.7368     33840

OOF Macro F1 (환자단위 소프트 보팅 최종 결합 성적): 0.6718


In [11]:
def slice_weight(s):
    return 1.5 if 15 <= s <= 45 else 1.0

test_ds_n = AlzheimerDataset(sub_df, TEST_DIR, transform=get_tta_transform(False), is_test=True)
test_ds_f = AlzheimerDataset(sub_df, TEST_DIR, transform=get_tta_transform(True), is_test=True)
test_loader_n = DataLoader(test_ds_n, batch_size=BATCH_SIZE*2, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader_f = DataLoader(test_ds_f, batch_size=BATCH_SIZE*2, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

test_sm_sum = np.zeros((len(sub_df), NUM_CLASSES), dtype=np.float32)
for fi in range(NUM_FOLDS):
    print(f"-> Fold {fi} 앙상블 모델 기반 17629장 추론 예측 작동 중...")
    model = build_model()
    model.load_state_dict(torch.load(CKPT_DIR / f"fold{fi}_best.pt", map_location=device))
    Ln = predict_logits(model, test_loader_n)
    Lf = predict_logits(model, test_loader_f)
    smn = F.softmax(torch.from_numpy(Ln), dim=1).numpy()
    smf = F.softmax(torch.from_numpy(Lf), dim=1).numpy()
    test_sm_sum += (smn + smf) / 2.0
    del model; torch.cuda.empty_cache(); gc.collect()

test_sm = test_sm_sum / NUM_FOLDS
print(f"슬라이스 단위 아그맥스(argmax) 기초 분포 카운트: {Counter(test_sm.argmax(1))}")


-> Fold 0 앙상블 모델 기반 17629장 추론 예측 작동 중...


-> Fold 1 앙상블 모델 기반 17629장 추론 예측 작동 중...
-> Fold 2 앙상블 모델 기반 17629장 추론 예측 작동 중...
-> Fold 3 앙상블 모델 기반 17629장 추론 예측 작동 중...
슬라이스 단위 아그맥스(argmax) 기초 분포 카운트: Counter({np.int64(2): 11732, np.int64(3): 4726, np.int64(0): 1154, np.int64(1): 17})


In [12]:
tp = sub_df.copy()
for c in range(NUM_CLASSES):
    tp[f'p{c}'] = test_sm[:, c]
tp['w'] = tp['slice_idx'].apply(slice_weight)
for c in range(NUM_CLASSES):
    tp[f'wp{c}'] = tp[f'p{c}'] * tp['w']

cols = [f'wp{c}' for c in range(NUM_CLASSES)]
pat_p = tp.groupby('patient_id')[cols].sum()
pat_w = tp.groupby('patient_id')['w'].sum()
pat_p = pat_p.div(pat_w, axis=0)
pat_label = pat_p.values.argmax(1)
pid2lbl = dict(zip(pat_p.index, pat_label))

tp['final'] = tp['patient_id'].map(pid2lbl)
tp['label'] = tp['final'].map(IDX2LABEL)

print("최종 보팅된 환자 단위 예측값 분포 카운트:", Counter(pat_label))
print("최종 마킹된 슬라이스 단위 분포 결과:")
print(tp['label'].value_counts())


최종 보팅된 환자 단위 예측값 분포 카운트: Counter({np.int64(2): 198, np.int64(3): 72, np.int64(0): 19})
최종 마킹된 슬라이스 단위 분포 결과:
label
NonDemented         12078
VeryMildDemented     4392
MildDemented         1159
Name: count, dtype: int64


In [13]:
sample_sub = pd.read_csv(SUB_CSV)
sub = sample_sub[['filename']].merge(tp[['filename', 'label']], on='filename', how='left')
assert sub['label'].notna().all(), "결측치 검출 에러 발생!"
sub.to_csv(OUTPUT_DIR / "submission_v2_2.csv", index=False)
print("🎯 [저장 완료] 환자 보팅 최종본 제출 파일 저장 경로:", OUTPUT_DIR / "submission_v2_2.csv")
print(sub['label'].value_counts())

sl = [IDX2LABEL[i] for i in test_sm.argmax(1)]
ssub = pd.DataFrame({'filename': sub_df['filename'], 'label': sl})
ssub = sample_sub[['filename']].merge(ssub, on='filename', how='left')
ssub.to_csv(OUTPUT_DIR / "submission_v2_slice.csv", index=False)
print("🎯 [저장 완료] 슬라이스 단독 대조군 제출 파일 저장 경로:", OUTPUT_DIR / "submission_v2_slice.csv")


🎯 [저장 완료] 환자 보팅 최종본 제출 파일 저장 경로: outputs\submission_v2_2.csv
label
NonDemented         12078
VeryMildDemented     4392
MildDemented         1159
Name: count, dtype: int64
🎯 [저장 완료] 슬라이스 단독 대조군 제출 파일 저장 경로: outputs\submission_v2_slice.csv
